In [ ]:
# --- Colab bootstrap: in Colab, clone + install the repo; locally, do nothing ---
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/CKmellow/per3-genotype-clock-gene-timing.git"
BRANCH = "main"  # change to your branch name to test it in Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    repo = Path("/content/per3-genotype-clock-gene-timing")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(repo)], check=True)
    os.chdir(repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    src = str(repo / "src")
else:  # local: find the repo root (works from notebooks/ or notebooks/stages/)
    here = Path.cwd().resolve()
    root = next((p for p in [here, *here.parents] if (p / "pyproject.toml").exists()), here)
    src = str(root / "src")
if src not in sys.path:
    sys.path.insert(0, src)  # makes `import per3clock` work without a kernel restart

from per3clock import config  # noqa: E402

config.set_seeds()
config.ensure_dirs()
print("per3clock root:", config.ROOT, "| Colab:", IN_COLAB)

# Stage 8 · Conclusions

*Owner: Part 1 (issue #6).*

## Minimum evidence (lecturer's brief)
*Brief, stage 8: "Answer the research question and state the limitations." Minimum evidence: "One point per objective, plus future steps. Upload to GitHub the code." Learning outcome 7: "Communicate results honestly, including negative or inconclusive findings."*

- [x] **Answer the research question** in one or two sentences
- [x] **One point per objective** (1–4), each citing the evidence (table/figure) it rests on
- [x] **Limitations**
- [x] **Future steps**
- [x] Negative or inconclusive findings stated honestly (no overclaiming from small n)
- [x] Code on GitHub; the final notebook runs top-to-bottom in Google Colab **with outputs saved**

## Contract
**Inputs:** tables written by stages 3–7 in `results/tables/` (`qc_counts.csv`, `acrophase_by_subject.csv`, `acrophase_genotype_tests.csv`, `acrophase_sensitivity_tests.csv`, `eda_subject_timing.csv`, `network_shift_test.csv`, `ml_phase_prediction.csv`, `ml_paired_comparisons.csv`). Nothing is re-estimated here. **Output:** figure `conclusions_genotype_shifts.png` (all genotype shifts on one axis, for the presentation).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from per3clock import circstats, config, plotting

plotting.set_style()
pd.set_option("display.width", 160)
T = config.TABLES_DIR
read = lambda name: pd.read_csv(T / f"{name}.csv")  # noqa: E731

## 8.1 Key numbers

**Data.** Sample flow from Stage 3:

In [ ]:
display(read("qc_counts")[["step", "samples", "subjects", "subjects_4_4", "subjects_5_5", "features"]])

**Objective 1: per-subject peak times** (Sleep Extension, clock time). Group peak time and how many people have a significant individual rhythm (cosinor p < 0.05):

In [ ]:
subj = read("acrophase_by_subject").query("sample_set == 'Primary'")
obj1 = (subj.groupby(["time_base", "gene"])
        .agg(n_subjects=("subject", "size"), n_rhythmic_p05=("p", lambda p: int((p < 0.05).sum())),
             median_amplitude=("amplitude", "median"),
             group_peak_h=("acrophase_h", lambda h: circstats.circular_mean_h(h)))
        .reset_index())
obj1["group_peak_clock"] = obj1["group_peak_h"].map(lambda h: f"{int(h % 24):02d}:{int(round(h % 1 * 60)) % 60:02d}")
display(obj1.round(2))

**Objectives 2 and 3: genotype shift in peak time** (5/5 − 4/4, hours; negative = 5/5 earlier). Primary analysis, 95 % bootstrap CI, permutation p and BH q:

In [ ]:
tests = read("acrophase_genotype_tests")
cols = ["gene", "time_base", "n_44", "n_55", "shift_h", "ci_lower_h", "ci_upper_h", "permutation_p", "permutation_q"]
display(tests[cols].round(2))
sens = read("acrophase_sensitivity_tests")
ci0 = ((sens.ci_lower_h <= 0) & (sens.ci_upper_h >= 0))
print(f"Sensitivity analyses: {ci0.sum()} of {len(sens)} CIs include 0; smallest q = {sens.permutation_q.min():.2f}")

net = read("network_shift_test")
display(net[["time_base", "n_genes", "n_subjects", "mean_shift_h", "ci_lower_h", "ci_upper_h", "n_earlier", "p_mean_shift"]].round(2))

timing = read("eda_subject_timing")
for col, what in [("wake_clock_h", "wake time"), ("phase0_clock_h", "melatonin phase 0")]:
    m = {g: circstats.circular_mean_h(timing.loc[timing.per3_genotype == g, col]) for g in config.GENOTYPES}
    print(f"{what}: 4/4 {m['4/4']:.2f} h, 5/5 {m['5/5']:.2f} h, 5/5 − 4/4 = {circstats.circular_diff_h(m['5/5'], m['4/4']):+.2f} h")

**Machine learning (Stage 7):** leave-one-subject-out error, and the paired comparisons:

In [ ]:
ml = read("ml_phase_prediction")
display(ml[["model", "median_abs_error_h", "median_ci95_low_h", "median_ci95_high_h", "pct_within_3h"]].round(2))
display(read("ml_paired_comparisons")[["model_a", "model_b", "mean_diff_h", "ci95_low_h", "ci95_high_h", "wilcoxon_p", "n_better", "n_subjects"]].round(3))

## 8.2 All genotype shifts on one axis

The answer to the research question in one picture: every estimate of how much earlier (or later) 5/5 carriers peak, with its 95 % CI.

In [ ]:
rows = [dict(label=f"{r.gene} · {'clock time' if r.time_base == 'clock_time' else 'circadian phase'}",
             shift=r.shift_h, lo=r.ci_lower_h, hi=r.ci_upper_h, kind="primary gene")
        for r in tests.itertuples()]
rows += [dict(label=f"14-gene clock network · {'clock time' if r.time_base == 'clock_time' else 'circadian phase'}",
              shift=r.mean_shift_h, lo=r.ci_lower_h, hi=r.ci_upper_h, kind="network")
         for r in net.itertuples()]
f = pd.DataFrame(rows)
f["axis"] = np.where(f.label.str.contains("clock time"), 0, 1)
f = f.sort_values(["axis", "kind"], ascending=[True, False]).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(8, 4.6), layout="constrained")
y = np.arange(len(f))[::-1]
ax.axvspan(-1.5, 1.5, color="0.92", zorder=0)
ax.axvline(0, color="0.35", lw=1)
col = np.where(f.axis == 0, "#3b6fb6", "#d9822b")
ax.hlines(y, f.lo, f.hi, color=col, lw=2)
ax.scatter(f["shift"], y, c=col, s=np.where(f.kind == "network", 90, 45),
           marker="o", zorder=3, edgecolor="k", linewidth=np.where(f.kind == "network", 1.2, 0))
ax.set_yticks(y, f.label)
ax.set_xlabel("Shift in peak time, 5/5 − 4/4 (h)   ← 5/5 earlier | 5/5 later →")
ax.set_title("PER3 genotype and clock-gene peak time: every 95% CI includes 0\n"
             "(blue: clock time; orange: melatonin-relative phase; grey band: ±half a 3-h sample interval)",
             fontsize=10)
plotting.save_fig(fig, "conclusions_genotype_shifts")

## Interpretation

### Answer to the research question
**We found no detectable shift in when PER2, PER3 and NR1D1 peak between PER3 4/4 and 5/5 carriers in human blood.** 5/5 carriers tend to peak about 0.5–1.4 h earlier on clock time, but every 95 % CI includes zero. Relative to each person's melatonin rhythm, the tendency disappears. With 11–13 people per group, this is an **inconclusive** result, not evidence of no effect.

### One point per objective
1. **Peak times (objective 1).** All three genes peak in the early morning, in the expected order: NR1D1 ~02:30, PER2 ~05:10, PER3 ~05:40 (clock time). That is about 1–1.5 h after the melatonin midpoint for PER2/PER3 and just before it for NR1D1. PER3's peak matches published leukocyte data (Archer et al. 2008, 06:05). Individual peak times are reliable for NR1D1 (13/24 people with a significant own rhythm) but noisy for PER3 (7/24). *Evidence: `acrophase_by_subject.csv`, Stage 5 rose plots, Stage 6 literature table.*
2. **Genotype comparison on clock time (objective 2).** 5/5 − 4/4 shift: NR1D1 −1.2 h [−3.1, +0.7], PER2 −0.5 h [−2.6, +1.7], PER3 −1.4 h [−6.3, +2.9]. All q = 0.66. In the 14-gene clock network, 12 of 14 genes peak earlier in 5/5 carriers (mean −1.0 h [−3.5, +0.6], p = 0.28). The direction is consistent but not significant. The data **cannot exclude** a 1–2 h earlier peak in 5/5 carriers. All 18 sensitivity-analysis CIs include zero (smallest q = 0.44). *Evidence: `acrophase_genotype_tests.csv`, `acrophase_sensitivity_tests.csv`, `network_shift_test.csv`, figure `conclusions_genotype_shifts.png`.*
3. **Whole clock vs clock genes (objective 3).** Measured against melatonin-based phase, the shifts shrink or reverse: NR1D1 +0.4 h, PER2 +1.1 h, PER3 −1.2 h, network −0.04 h [−1.8, +1.1]. The 5/5 group also woke 1.2 h earlier and had its melatonin phase 0 1.25 h earlier. That is about the size of their (non-significant) clock-time head start. So whatever earliness there is belongs to the **whole sleep–circadian system**, not to the clock genes relative to the internal clock. One caveat: in Sleep Extension the phase axis is also time since waking, so we cannot say whether the shared shift is driven by the clock or by wake time. Stage 7 adds a consistent finding: PER3 genotype adds no information when predicting phase from expression (−0.01 h, p = 0.69).
4. **Hypothesis (objective 4).** The hypothesis that 5/5 carriers reach peak clock-gene expression earlier **is not supported** at the 5 % level. The estimates point in the predicted direction on clock time, so the result is compatible with a modest effect that this sample is too small to detect. It is **not** compatible with a large one: shifts of more than ~3 h are excluded for NR1D1 and PER2. This agrees with Viola et al. (2007), who found no PER3-genotype effect on PER3 mRNA timing in a similar-size sample.

**Supporting results.** Blood has a strong daily transcriptional programme: 2,007 genes are rhythmic at q < 0.05 with amplitude ≥ 0.2 (published: 1,855). It is dominated by white-cell traffic: naive T cells peak at night and NK cells and monocytes by day (Stage 6). Internal phase can be predicted from expression with about 1 h median error for a new person. Expression adds a small but consistent 0.4 h over clock time, and only after each person's baseline is removed (Stage 7).

### Limitations
- **Small groups:** 11 vs 13 subjects (11 vs 12 on circadian phase), so CIs are 2–9 h wide. Non-significant ≠ no effect.
- **Coarse timing:** blood every 3 h and circadian phase in 3-h bins. Shifts under ~1.5 h are below the sampling resolution, and individual PER3 rhythms are weak (7/24 significant).
- **Phase is wake-relative in Sleep Extension:** `circadian_phase_h` = hours awake − 19.5 for every analysed subject (`data/README.md`, Known issues 9). Objective 3 and the ML target cannot separate the internal clock from time since waking.
- **Homozygotes only, one cohort:** only 4/4 and 5/5 (no 4/5), all from one study and one dataset. GSE48113 has no phase labels, so there is no external validation.
- **Blood, not the brain clock:** peripheral leukocyte rhythms mix intracellular clocks with daily changes in white-cell composition. We had no cell counts to separate them.
- **Unusual conditions:** samples were taken during ~40 h of total sleep deprivation after a controlled sleep week, not on a normal day. Sleep Restriction was only a sensitivity analysis.
- **Phase 0 definition** (≈ plasma-melatonin midpoint) is inferred from the paper's reported means, not confirmed in its SI Methods.

### Future steps
1. **Power a confirmatory study.** Our between-person circular SDs of peak time are ~1.7–2.5 h for NR1D1 and PER2 and ~4–4.6 h for PER3. A normal-approximation calculation (80 % power, two-sided α = 0.05) gives roughly **45–100 people per genotype** to detect a 1-h shift in NR1D1/PER2, and over 200 for PER3. This is a rough estimate, not a formal circular power analysis. Include 4/5 heterozygotes for a dose test.
2. **Denser sampling** (every 1–2 h) and continuous melatonin phase (DLMO), so that sub-bin shifts become measurable and phase is independent of wake time.
3. **Separate cell composition from intracellular clocks:** deconvolve whole-blood expression or use sorted monocytes/T cells.
4. **External replication** in another genotyped time-series cohort with melatonin phase. Test the Stage 7 phase model there as well, ideally from one or two blood draws rather than a full series.
5. **Normal sleep conditions:** repeat outside sleep deprivation to check that the timing is not set by the extended-wake protocol.